Get the per-minute price from Wharton Research Data Services (WRDS). Relevant documentation:

- https://wrds-jupyter.wharton.upenn.edu/user/afeld/lab/tree/wrds-code-samples/taq/python/minute_price.ipynb
- https://wrds-www.wharton.upenn.edu/pages/support/programming-wrds/programming-python/python-from-your-computer/
- https://wrds-www.wharton.upenn.edu/pages/support/programming-wrds/programming-python/querying-wrds-data-python/
- https://wrds-www.wharton.upenn.edu/pages/support/manuals-and-overviews/taq/general/wrds-overview-taq/
- https://wrds-www.wharton.upenn.edu/documents/1443/wrds_connection.html


In [1]:
import wrds

conn = wrds.Connection()
conn

Loading library list...
Done


In [2]:
sorted(conn.list_libraries())

['aha_sample',
 'ahasamp',
 'auditsmp',
 'auditsmp_all',
 'bank',
 'bank_all',
 'bank_premium_samp',
 'banksamp',
 'block',
 'block_all',
 'boardex_trial',
 'boardsmp',
 'bvd_amadeus_trial',
 'bvd_bvdbankf_trial',
 'bvd_orbis_trial',
 'bvdsamp',
 'calcbench_trial',
 'calcbnch',
 'candid_samp',
 'cboe',
 'cboe_all',
 'cboe_sample',
 'cboesamp',
 'cddsamp',
 'ciq',
 'ciq_capstrct',
 'ciq_common',
 'ciq_keydev',
 'ciq_pplintel',
 'ciq_transcripts',
 'ciqsamp',
 'ciqsamp_capstrct',
 'ciqsamp_common',
 'ciqsamp_keydev',
 'ciqsamp_pplintel',
 'ciqsamp_ratings',
 'ciqsamp_transactions',
 'ciqsamp_transcripts',
 'cisdmsmp',
 'columnar',
 'comp',
 'comp_bank_daily',
 'comp_execucomp',
 'comp_filings',
 'comp_global_daily',
 'comp_na_daily_all',
 'comp_segments_hist_daily',
 'compsamp',
 'compsamp_all',
 'compsamp_computext',
 'compsamp_snapshot',
 'compseg',
 'contrib',
 'contrib_bond_dickerson',
 'contrib_bond_firm_link',
 'contrib_ceo_turnover',
 'contrib_char_returns',
 'contrib_corp_fed_lit

In [3]:
LIBRARY = "taqm_2024"

conn.list_tables(LIBRARY)

['complete_nbbo_2024',
 'complete_nbbo_20240102',
 'complete_nbbo_20240103',
 'complete_nbbo_20240104',
 'complete_nbbo_20240105',
 'complete_nbbo_20240108',
 'complete_nbbo_20240109',
 'complete_nbbo_20240110',
 'complete_nbbo_20240111',
 'complete_nbbo_20240112',
 'complete_nbbo_20240116',
 'complete_nbbo_20240117',
 'complete_nbbo_20240118',
 'complete_nbbo_20240119',
 'complete_nbbo_20240122',
 'complete_nbbo_20240123',
 'complete_nbbo_20240124',
 'complete_nbbo_20240125',
 'complete_nbbo_20240126',
 'complete_nbbo_20240129',
 'complete_nbbo_20240130',
 'complete_nbbo_20240131',
 'complete_nbbo_20240201',
 'complete_nbbo_20240202',
 'complete_nbbo_20240205',
 'complete_nbbo_20240206',
 'complete_nbbo_20240207',
 'complete_nbbo_20240208',
 'complete_nbbo_20240209',
 'complete_nbbo_20240212',
 'complete_nbbo_20240213',
 'complete_nbbo_20240214',
 'complete_nbbo_20240215',
 'complete_nbbo_20240216',
 'complete_nbbo_20240220',
 'complete_nbbo_20240221',
 'complete_nbbo_20240222',
 'com

In [4]:
conn.describe_table(LIBRARY, "ctm_20241203")

Approximately 80411896 rows in taqm_2024.ctm_20241203.


,name,nullable,type,comment
0,date,False,DATE,None
1,time_m,False,TIME,None
2,time_m_nano,True,SMALLINT,None
3,ex,True,VARCHAR(1),None
4,sym_root,False,TEXT,None
5,sym_suffix,True,TEXT,None
6,tr_scond,True,VARCHAR(4),None
7,size,True,INTEGER,None
8,price,True,NUMERIC,None
9,tr_stop_ind,True,VARCHAR(1),None


In [5]:
df = conn.raw_sql(f"""
WITH milli AS (
    SELECT * FROM {LIBRARY}.ctm_20241203
    UNION ALL
    SELECT * FROM {LIBRARY}.ctm_20241204
)
SELECT
    date_bin('1 minute', date + time_m, TIMESTAMP '2024-12-03') AS ts_minute,
    AVG(price) AS average_price
FROM milli
WHERE sym_root = 'UNH'
GROUP BY ts_minute
ORDER BY ts_minute
""")

df

,ts_minute,average_price
0,2024-12-03 04:18:00,608.996667
1,2024-12-03 04:31:00,608.99
2,2024-12-03 05:07:00,608.52
3,2024-12-03 05:30:00,608.31
4,2024-12-03 06:42:00,608.51
...,...,...
1199,2024-12-04 19:53:00,610.08
1200,2024-12-04 19:54:00,610.08
1201,2024-12-04 19:55:00,612.9
1202,2024-12-04 19:58:00,611.84


In [6]:
df.to_csv("minute_price.csv", index=False)